# Dashboard combinado: EDA Compresores + Telemetria Industrial — Reto 8

Notebook unificado con dos grandes modulos:

- **Tab 1 — EDA Compresores A-D**: analisis exploratorio interactivo de los cuatro compresores con filtros dinamicos (paleta clara LANIT).
- **Tab 2 — Telemetria Industrial**: dashboard de telemetria de red con tema oscuro y 6 sub-pestanas.

Ejecuta todas las celdas en orden y abre **http://localhost:8058**

In [ ]:
# Descomenta si falta alguna libreria:
# import sys
# !{sys.executable} -m pip install pandas numpy plotly dash scipy statsmodels kaleido -q

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, dash_table

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Configuracion

Rutas de archivos, paletas de colores LANIT y estilos para ambos modulos.

In [ ]:
# ======================================================
#  PALETA LANIT CONSULTING
# ======================================================
LANIT = {
    "rojo":        "#D40000",
    "rojo_oscuro": "#A30000",
    "gris_oscuro": "#555555",
    "gris_medio":  "#8A8A8A",
    "gris_claro":  "#D9D9D9",
    "blanco":      "#FFFFFF",
}

# ======================================================
#  MODULO 1 — COMPRESORES (tema claro)
# ======================================================
CSV_PATHS = {
    "Compresor A": r"C:\25_26_R8_Equipo_Naranja\Datos\Originales\CompA.csv",
    "Compresor B": r"C:\25_26_R8_Equipo_Naranja\Datos\Originales\CompB.csv",
    "Compresor C": r"C:\25_26_R8_Equipo_Naranja\Datos\Originales\CompC.csv",
    "Compresor D": r"C:\25_26_R8_Equipo_Naranja\Datos\Originales\CompD.csv",
}

OUTPUT_DIR = Path(r"C:\Users\arbai\OneDrive - Mondragon Unibertsitatea\Escritorio\BDA2\reto8\plantilla_latex\eda_figures")
if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = Path("eda_figures")
OUTPUT_DIR.mkdir(exist_ok=True)

COMP_COLORS = {
    "Compresor A": "#D40000",
    "Compresor B": "#A30000",
    "Compresor C": "#555555",
    "Compresor D": "#8A8A8A",
}
COMP_COLORS_RGBA = {
    "Compresor A": "rgba(212,0,0,0.12)",
    "Compresor B": "rgba(163,0,0,0.12)",
    "Compresor C": "rgba(85,85,85,0.12)",
    "Compresor D": "rgba(138,138,138,0.12)",
}
TEMPLATE = "plotly_white"
FONT     = dict(family="Arial", size=13)

# ======================================================
#  MODULO 2 — TELEMETRIA (tema oscuro con LANIT)
# ======================================================
TELEM_CSV = r"C:\25_26_R8_Equipo_Naranja\Datos\Transformados\resultados_consultas.csv"

COLORS = {
    "bg":      "#0d1117",
    "card":    "#161b22",
    "border":  "#30363d",
    "accent1": "#D40000",
    "accent2": "#A30000",
    "accent3": "#D9D9D9",
    "accent4": "#8A8A8A",
    "accent5": "#555555",
    "text":    "#e6edf3",
    "subtext": "#8b949e",
}

PLOTLY_TEMPLATE = dict(
    layout=dict(
        paper_bgcolor=COLORS["card"],
        plot_bgcolor=COLORS["bg"],
        font=dict(color=COLORS["text"], family="Arial, sans-serif", size=12),
        xaxis=dict(gridcolor=COLORS["border"], zerolinecolor=COLORS["border"]),
        yaxis=dict(gridcolor=COLORS["border"], zerolinecolor=COLORS["border"]),
        legend=dict(bgcolor=COLORS["card"], bordercolor=COLORS["border"]),
        margin=dict(t=50, b=40, l=50, r=20),
    )
)

CARD_STYLE = {
    "backgroundColor": COLORS["card"],
    "border": f"1px solid {COLORS['border']}",
    "borderRadius": "8px",
    "padding": "20px",
    "marginBottom": "16px",
}
KPI_STYLE = {
    **CARD_STYLE,
    "textAlign": "center",
    "flex": "1",
    "minWidth": "140px",
    "marginBottom": "0",
}
PAGE_STYLE = {
    "backgroundColor": COLORS["bg"],
    "minHeight": "100vh",
    "padding": "24px 32px",
    "fontFamily": "Arial, sans-serif",
    "color": COLORS["text"],
}

print("Configuracion cargada")
print(f"Figuras PNG -> {OUTPUT_DIR}")

## 2. Datos de los compresores

Carga, limpieza y variables derivadas para los cuatro compresores.

In [ ]:
# Carga
dfs = []
for comp, path in CSV_PATHS.items():
    tmp = pd.read_csv(path, low_memory=False)
    tmp.columns = tmp.columns.str.strip()
    tmp["Compresor"] = comp
    dfs.append(tmp)

df_comp = pd.concat(dfs, ignore_index=True)
print(f"Compresores - dataset unido: {df_comp.shape}")
display(df_comp.head(3))

In [ ]:
# Limpieza y variables derivadas
num_cols = ["Presion", "Temperatura", "Frecuencia", "Potencia_Medida", "Potencia_Estimada"]
for col in num_cols:
    df_comp[col] = pd.to_numeric(df_comp[col], errors="coerce")

df_comp = df_comp.dropna(subset=num_cols + ["Compresor"]).copy()
df_comp = df_comp[
    (df_comp["Frecuencia"]        >= 0) &
    (df_comp["Potencia_Medida"]   >= 0) &
    (df_comp["Potencia_Estimada"] >= 0)
].copy()

df_comp["Error_Potencia"] = df_comp["Potencia_Medida"] - df_comp["Potencia_Estimada"]
df_comp["Error_Abs"]      = df_comp["Error_Potencia"].abs()
df_comp["Error_Pct"]      = np.where(
    df_comp["Potencia_Estimada"].abs() > 0,
    df_comp["Error_Potencia"] / df_comp["Potencia_Estimada"] * 100,
    np.nan,
)
df_comp["Estado"] = np.where(df_comp["Frecuencia"] <= 1, "Parado", "Funcionando")

# Deteccion de anomalias por IQR
df_comp["Anomalia"] = False
for comp in df_comp["Compresor"].unique():
    mask   = df_comp["Compresor"] == comp
    q1, q3 = df_comp.loc[mask, "Error_Abs"].quantile([0.25, 0.75])
    iqr    = q3 - q1
    df_comp.loc[mask, "Anomalia"] = df_comp.loc[mask, "Error_Abs"] > (q3 + 1.5 * iqr)

df_comp["Frec_Bin"] = pd.cut(df_comp["Frecuencia"], bins=np.arange(0, 105, 5), include_lowest=True)

# Dataset visual (recorte de outliers extremos)
cols_clip = ["Presion","Temperatura","Frecuencia","Potencia_Medida","Potencia_Estimada","Error_Potencia","Error_Abs"]
vdf_comp = df_comp.copy()
for col in cols_clip:
    lo, hi = vdf_comp[col].quantile([0.005, 0.995])
    vdf_comp = vdf_comp[(vdf_comp[col] >= lo) & (vdf_comp[col] <= hi)]

print(f"Compresores - limpio: {df_comp.shape}")
print(f"Compresores - visual: {vdf_comp.shape}")

## 3. Datos de telemetria

Carga y preprocesado del CSV de resultados de consultas de red.

In [ ]:
# Carga telemetria
df = pd.read_csv(TELEM_CSV)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df["indice"]    = range(len(df))
df["rango_temp"] = df["temperatura_maxima"] - df["temperatura_minima"]
df["sesion"]    = df["timestamp"].dt.strftime("%Y-%m-%d %H:%M")

if len(df) > 1000:
    step = len(df) // 800
    df_line_sample = df.iloc[::step].copy()
else:
    df_line_sample = df.copy()

print(f"Telemetria - total registros: {len(df)}")
print(f"Telemetria - muestra graficos: {len(df_line_sample)} puntos")
print(f"Rango temperaturas: {df['media_temperatura'].min():.1f} - {df['media_temperatura'].max():.1f} C")
display(df.head(3))

## 4. Funciones auxiliares de estilo y exportacion

In [ ]:
# Helpers compresores
def style(fig, title="", height=460):
    fig.update_layout(
        template=TEMPLATE, font=FONT, height=height,
        title=dict(text=f"<b>{title}</b>", x=0.03, font=dict(size=14, color="#D40000")),
        margin=dict(l=55, r=25, t=65, b=55),
        legend=dict(title_text="", bgcolor="rgba(255,255,255,0.85)",
                    bordercolor="#ddd", borderwidth=1),
        plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_xaxes(showgrid=True, gridcolor="#f0f0f0", linecolor="#ccc", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0", linecolor="#ccc", zeroline=False)
    return fig

def save_png(fig, name):
    path = OUTPUT_DIR / f"{name}.png"
    try:
        fig.write_image(str(path), width=1400, height=700, scale=2)
        print(f"  OK  {path}")
    except Exception as e:
        print(f"  ERROR  {name}: {e}  (kaleido no instalado?)")

# Helpers telemetria
def apply_template(fig):
    fig.update_layout(**PLOTLY_TEMPLATE["layout"])
    return fig

def kpi_card(title, value, unit="", color=None):
    if color is None:
        color = COLORS["accent1"]
    return html.Div([
        html.P(title, style={"color": COLORS["subtext"], "fontSize": "11px",
                              "margin": "0 0 6px 0", "textTransform": "uppercase",
                              "letterSpacing": "1px"}),
        html.Span(f"{value}", style={"fontSize": "26px", "fontWeight": "700", "color": color}),
        html.Span(f" {unit}", style={"fontSize": "13px", "color": COLORS["subtext"]}),
    ], style=KPI_STYLE)

print("Helpers cargados")

## 5. Funciones de figuras — Compresores

Cada funcion acepta un DataFrame `data` y devuelve una figura Plotly estilizada con la paleta LANIT.

In [ ]:
def fig_media_potencias(data):
    media = (
        data.groupby("Compresor")[["Potencia_Medida", "Potencia_Estimada"]]
        .mean().reset_index()
        .melt(id_vars="Compresor", var_name="Tipo", value_name="Potencia media (W)")
    )
    fig = px.bar(
        media, x="Compresor", y="Potencia media (W)", color="Tipo",
        barmode="group", text_auto=".1f",
        color_discrete_sequence=["#D40000", "#A30000"],
    )
    fig.update_traces(textposition="outside")
    return style(fig, "Potencia media: medida vs estimada")

In [ ]:
def fig_scatter_medida_estimada(data):
    s = data.sample(min(4000, len(data)), random_state=42)
    fig = px.scatter(
        s, x="Potencia_Estimada", y="Potencia_Medida",
        color="Compresor", color_discrete_map=COMP_COLORS, opacity=0.22,
        labels={"Potencia_Estimada": "Potencia estimada (W)",
                "Potencia_Medida": "Potencia medida (W)"},
    )
    lo = min(s["Potencia_Estimada"].min(), s["Potencia_Medida"].min())
    hi = max(s["Potencia_Estimada"].max(), s["Potencia_Medida"].max())
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[lo, hi], mode="lines", name="Linea ideal",
        line=dict(color="#333", dash="dash", width=2),
    ))
    return style(fig, "Potencia medida vs potencia estimada")

In [ ]:
def fig_error_violin(data):
    fig = px.violin(
        data, x="Compresor", y="Error_Potencia", color="Compresor",
        color_discrete_map=COMP_COLORS, box=True, points=False,
        labels={"Error_Potencia": "Error de potencia (W)"},
    )
    fig.add_hline(
        y=0, line_dash="dash", line_color="#555",
        annotation_text="Error = 0", annotation_position="top right",
    )
    return style(fig, "Distribucion del error de potencia por compresor")

In [ ]:
def fig_tendencia_frecuencia(data):
    tend = (
        data.dropna(subset=["Frec_Bin"])
        .groupby(["Frec_Bin", "Compresor"])
        .agg(
            Media=("Potencia_Medida", "mean"),
            Q25  =("Potencia_Medida", lambda x: x.quantile(0.25)),
            Q75  =("Potencia_Medida", lambda x: x.quantile(0.75)),
        )
        .reset_index()
    )
    tend["Frec_Bin"] = tend["Frec_Bin"].astype(str)
    fig = go.Figure()
    for comp, color in COMP_COLORS.items():
        d = tend[tend["Compresor"] == comp]
        if d.empty:
            continue
        x_band = list(d["Frec_Bin"]) + list(d["Frec_Bin"])[::-1]
        y_band = list(d["Q75"])      + list(d["Q25"])[::-1]
        fig.add_trace(go.Scatter(
            x=x_band, y=y_band, fill="toself",
            fillcolor=COMP_COLORS_RGBA[comp],
            line=dict(color="rgba(255,255,255,0)"),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=d["Frec_Bin"], y=d["Media"], name=comp,
            mode="lines+markers",
            line=dict(color=color, width=2),
            marker=dict(size=6),
        ))
    fig.update_xaxes(tickangle=45)
    return style(fig, "Tendencia media de potencia por rango de frecuencia (banda IQR 25-75 %)")

In [ ]:
def fig_distribucion_potencia(data):
    fig = px.histogram(
        data, x="Potencia_Medida", color="Compresor",
        color_discrete_map=COMP_COLORS, nbins=60,
        barmode="overlay", opacity=0.50,
        labels={"Potencia_Medida": "Potencia medida (W)"},
    )
    return style(fig, "Distribucion de la potencia medida por compresor")

In [ ]:
def fig_anomalias(data):
    anom = (
        data.groupby("Compresor")["Anomalia"]
        .mean().mul(100).reset_index(name="% Anomalias")
    )
    fig = px.bar(
        anom, x="Compresor", y="% Anomalias", color="Compresor",
        color_discrete_map=COMP_COLORS,
        text=anom["% Anomalias"].map(lambda v: f"{v:.3f} %"),
    )
    fig.update_traces(textposition="outside")
    return style(fig, "Porcentaje de anomalias detectadas por compresor (criterio IQR)")

In [ ]:
def fig_correlacion(data):
    cols = ["Presion","Temperatura","Frecuencia",
            "Potencia_Medida","Potencia_Estimada",
            "Error_Potencia","Error_Abs"]
    corr = data[cols].corr().round(2)
    fig = px.imshow(
        corr, text_auto=".2f",
        color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
        labels=dict(color="r"),
    )
    fig.update_xaxes(tickangle=45)
    return style(fig, "Matriz de correlacion de Pearson", height=520)

In [ ]:
def fig_presion_temp(data):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Presion por compresor (atm)", "Temperatura por compresor (C)"],
    )
    for i, col in enumerate(["Presion", "Temperatura"], 1):
        for comp, color in COMP_COLORS.items():
            d = data[data["Compresor"] == comp]
            fig.add_trace(
                go.Box(y=d[col], name=comp, marker_color=color, showlegend=(i == 1)),
                row=1, col=i,
            )
    return style(fig, "Variables ambientales: presion y temperatura")

In [ ]:
def fig_frecuencia_hist(data):
    fig = px.histogram(
        data, x="Frecuencia", color="Compresor",
        color_discrete_map=COMP_COLORS, nbins=50,
        barmode="overlay", opacity=0.50,
        labels={"Frecuencia": "Frecuencia (%)"},
    )
    return style(fig, "Distribucion de frecuencia de operacion")

In [ ]:
def fig_frecuencia_potencia(data):
    activos = data[data["Frecuencia"] > 1]
    s = activos.sample(min(4000, len(activos)), random_state=42)
    try:
        fig = px.scatter(
            s, x="Frecuencia", y="Potencia_Medida",
            color="Compresor", color_discrete_map=COMP_COLORS,
            opacity=0.20, trendline="lowess",
            labels={"Frecuencia": "Frecuencia (%)",
                    "Potencia_Medida": "Potencia medida (W)"},
        )
    except Exception:
        fig = px.scatter(
            s, x="Frecuencia", y="Potencia_Medida",
            color="Compresor", color_discrete_map=COMP_COLORS, opacity=0.20,
            labels={"Frecuencia": "Frecuencia (%)",
                    "Potencia_Medida": "Potencia medida (W)"},
        )
    return style(fig, "Relacion frecuencia - potencia con tendencia LOWESS")

In [ ]:
def compute_summary(data):
    return (
        data.groupby("Compresor")
        .agg(
            N              =("Compresor",      "size"),
            Frec_media     =("Frecuencia",      "mean"),
            Frec_std       =("Frecuencia",      "std"),
            Potencia_media =("Potencia_Medida", "mean"),
            Potencia_std   =("Potencia_Medida", "std"),
            Error_medio    =("Error_Potencia",  "mean"),
            Error_abs_medio=("Error_Abs",       "mean"),
            Presion_media  =("Presion",         "mean"),
            Temp_media     =("Temperatura",     "mean"),
            Anomalias_n    =("Anomalia",        "sum"),
            Anomalias_pct  =("Anomalia", lambda x: round(x.mean()*100, 3)),
        )
        .reset_index()
        .round(3)
    )

print("Funciones de compresores listas")

## 6. Exportar figuras a PNG

Genera los 10 PNG en `plantilla_latex/eda_figures/` para el informe LaTeX.

Requiere `pip install kaleido`

In [ ]:
print("Generando y guardando figuras...\n")

_figures = {
    "fig_media_potencias":         fig_media_potencias(vdf_comp),
    "fig_scatter_medida_estimada": fig_scatter_medida_estimada(vdf_comp),
    "fig_error_violin":            fig_error_violin(vdf_comp),
    "fig_tendencia_frecuencia":    fig_tendencia_frecuencia(vdf_comp),
    "fig_distribucion_potencia":   fig_distribucion_potencia(vdf_comp),
    "fig_anomalias":               fig_anomalias(df_comp),
    "fig_correlacion":             fig_correlacion(vdf_comp),
    "fig_presion_temperatura":     fig_presion_temp(vdf_comp),
    "fig_frecuencia_hist":         fig_frecuencia_hist(vdf_comp),
    "fig_frecuencia_potencia":     fig_frecuencia_potencia(vdf_comp),
}
for name, fig in _figures.items():
    save_png(fig, name)

print(f"\n{len(_figures)} figuras procesadas en {OUTPUT_DIR}")

## 7. Funciones de paginas — Telemetria

Cada funcion genera el contenido completo de una sub-pestana del modulo de telemetria.

In [ ]:
def page_resumen():
    total_registros = len(df)
    temp_media      = df["media_temperatura"].mean()
    potencia_media  = df["potencia_media"].mean()
    potencia_max    = df["potencia_maxima"].max()
    alertas_max     = df["alertas_potencia"].max()
    sesiones        = df["timestamp"].dt.date.nunique()

    fig_overview = make_subplots(specs=[[{"secondary_y": True}]])
    fig_overview.add_trace(go.Scatter(
        x=df_line_sample["indice"], y=df_line_sample["potencia_media"],
        name="Potencia media (W)", line=dict(color=COLORS["accent1"], width=1.5), opacity=0.8,
    ), secondary_y=False)
    fig_overview.add_trace(go.Scatter(
        x=df_line_sample["indice"], y=df_line_sample["media_temperatura"],
        name="Temperatura media (C)", line=dict(color=COLORS["accent3"], width=1.5, dash="dot"), opacity=0.8,
    ), secondary_y=True)
    apply_template(fig_overview)
    fig_overview.update_layout(title="Evolucion de Potencia y Temperatura", height=400,
                                xaxis_title="Numero de registro (muestreado)")
    fig_overview.update_yaxes(title_text="Potencia (W)", secondary_y=False)
    fig_overview.update_yaxes(title_text="Temperatura (C)", secondary_y=True)

    fig_hist = px.histogram(df, x="potencia_media", nbins=40,
                             title="Distribucion de Potencia Media",
                             labels={"potencia_media": "Potencia (W)", "count": "Frecuencia"})
    fig_hist.update_traces(marker_color=COLORS["accent1"], opacity=0.7)
    apply_template(fig_hist)
    fig_hist.update_layout(height=300)

    df_corr_sample = df.sample(min(800, len(df)))
    fig_corr = px.scatter(df_corr_sample, x="media_temperatura", y="potencia_media",
                           color="frecuencia_media", color_continuous_scale="Reds",
                           title="Correlacion: Temperatura vs Potencia")
    apply_template(fig_corr)
    fig_corr.update_layout(height=300)

    return html.Div([
        html.H2("Resumen del Sistema", style={"margin": "0 0 20px 0"}),
        html.Div([
            kpi_card("Registros",        f"{total_registros:,}",    "",    COLORS["accent1"]),
            kpi_card("Temp. media",      f"{temp_media:.1f}",       "C",   COLORS["accent3"]),
            kpi_card("Potencia media",   f"{potencia_media:.0f}",   "W",   COLORS["accent1"]),
            kpi_card("Potencia maxima",  f"{potencia_max:.0f}",     "W",   COLORS["accent2"]),
            kpi_card("Alertas max.",     f"{alertas_max}",          "",    COLORS["accent2"]),
            kpi_card("Sesiones",         f"{sesiones}",             "dias",COLORS["accent4"]),
        ], style={"display":"flex","gap":"12px","flexWrap":"wrap","marginBottom":"24px"}),
        html.Div([dcc.Graph(figure=fig_overview)], style=CARD_STYLE),
        html.Div([
            html.Div([dcc.Graph(figure=fig_hist)], style={**CARD_STYLE,"flex":"1","marginRight":"8px"}),
            html.Div([dcc.Graph(figure=fig_corr)], style={**CARD_STYLE,"flex":"1","marginLeft":"8px"}),
        ], style={"display":"flex","gap":"16px"}),
    ], style=PAGE_STYLE)

In [ ]:
def page_temperatura():
    fig_line = go.Figure()
    fig_line.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["media_temperatura"],
        name="Media", line=dict(color=COLORS["accent1"], width=2), mode="lines"))
    fig_line.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["temperatura_maxima"],
        name="Maxima", line=dict(color=COLORS["accent2"], width=1.5, dash="dash"), mode="lines"))
    fig_line.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["temperatura_minima"],
        name="Minima", line=dict(color=COLORS["accent3"], width=1.5, dash="dot"), mode="lines"))
    apply_template(fig_line)
    fig_line.update_layout(title="Evolucion de Temperaturas", height=420,
        xaxis_title="Numero de registro (muestreado)", yaxis_title="Temperatura (C)", hovermode="x unified")

    fig_rango = go.Figure()
    fig_rango.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["rango_temp"],
        mode="lines", fill="tozeroy", line=dict(color=COLORS["accent4"], width=2), name="Rango (Max-Min)"))
    apply_template(fig_rango)
    fig_rango.update_layout(title="Rango Termico", height=280,
        xaxis_title="Numero de registro", yaxis_title="Variacion (C)")

    fig_box = go.Figure()
    fig_box.add_trace(go.Box(y=df["temperatura_minima"], name="Minima", marker_color=COLORS["accent3"], boxmean=True))
    fig_box.add_trace(go.Box(y=df["media_temperatura"],  name="Media",  marker_color=COLORS["accent1"], boxmean=True))
    fig_box.add_trace(go.Box(y=df["temperatura_maxima"], name="Maxima", marker_color=COLORS["accent2"], boxmean=True))
    apply_template(fig_box)
    fig_box.update_layout(title="Distribucion de Temperaturas", height=320, yaxis_title="Temperatura (C)")

    return html.Div([
        html.H2("Analisis de Temperatura", style={"margin": "0 0 20px 0"}),
        html.Div([
            kpi_card("Temp. media",  f"{df['media_temperatura'].mean():.1f}", "C", COLORS["accent1"]),
            kpi_card("Temp. maxima", f"{df['temperatura_maxima'].max():.1f}", "C", COLORS["accent2"]),
            kpi_card("Temp. minima", f"{df['temperatura_minima'].min():.1f}", "C", COLORS["accent3"]),
            kpi_card("Rango maximo", f"{df['rango_temp'].max():.1f}",         "C", COLORS["accent4"]),
        ], style={"display":"flex","gap":"12px","flexWrap":"wrap","marginBottom":"24px"}),
        html.Div([dcc.Graph(figure=fig_line)], style=CARD_STYLE),
        html.Div([
            html.Div([dcc.Graph(figure=fig_rango)], style={**CARD_STYLE,"flex":"1"}),
            html.Div([dcc.Graph(figure=fig_box)],   style={**CARD_STYLE,"flex":"1"}),
        ], style={"display":"flex","gap":"16px"}),
    ], style=PAGE_STYLE)

In [ ]:
def page_potencia():
    fig_pot = go.Figure()
    fig_pot.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["potencia_media"],
        name="Potencia Media", line=dict(color=COLORS["accent1"], width=2), fill="tozeroy"))
    fig_pot.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["potencia_maxima"],
        name="Potencia Maxima", line=dict(color=COLORS["accent2"], width=1.5, dash="dash")))
    apply_template(fig_pot)
    fig_pot.update_layout(title="Evolucion de Potencia", height=400,
        xaxis_title="Numero de registro", yaxis_title="Potencia (W)", hovermode="x unified")

    fig_comp_pot = go.Figure()
    fig_comp_pot.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["comparacion_potencias"],
        name="Delta Potencia", line=dict(color=COLORS["accent2"], width=2), fill="tozeroy"))
    fig_comp_pot.add_hline(y=0, line_dash="dash", line_color=COLORS["subtext"], opacity=0.5)
    apply_template(fig_comp_pot)
    fig_comp_pot.update_layout(title="Comparacion de Potencias", height=280,
        xaxis_title="Numero de registro", yaxis_title="Diferencia (W)")

    df_alert_high = df[df["alertas_potencia"] > 0]
    fig_scatter = go.Figure()
    normal_sample = df_line_sample[df_line_sample["alertas_potencia"] == 0]
    if len(normal_sample) > 500:
        normal_sample = normal_sample.sample(500)
    fig_scatter.add_trace(go.Scatter(x=normal_sample["indice"], y=normal_sample["potencia_media"],
        mode="markers", marker=dict(size=3, color=COLORS["accent4"], opacity=0.4), name="Normal"))
    if len(df_alert_high) > 0:
        alert_sample = df_alert_high if len(df_alert_high) <= 300 else df_alert_high.sample(300)
        fig_scatter.add_trace(go.Scatter(x=alert_sample["indice"], y=alert_sample["potencia_media"],
            mode="markers", marker=dict(size=7, color=COLORS["accent2"], symbol="x"), name="Con Alertas"))
    apply_template(fig_scatter)
    fig_scatter.update_layout(title="Puntos con Alertas de Potencia", height=300,
        xaxis_title="Numero de registro", yaxis_title="Potencia Media (W)")

    return html.Div([
        html.H2("Analisis de Potencia", style={"margin": "0 0 20px 0"}),
        html.Div([
            kpi_card("Potencia media",      f"{df['potencia_media'].mean():.1f}",      "W", COLORS["accent1"]),
            kpi_card("Potencia maxima",     f"{df['potencia_maxima'].max():.1f}",      "W", COLORS["accent2"]),
            kpi_card("Comparacion media",   f"{df['comparacion_potencias'].mean():.2f}","W", COLORS["accent3"]),
            kpi_card("Desviacion std",      f"{df['potencia_media'].std():.1f}",       "W", COLORS["accent4"]),
        ], style={"display":"flex","gap":"12px","flexWrap":"wrap","marginBottom":"24px"}),
        html.Div([dcc.Graph(figure=fig_pot)], style=CARD_STYLE),
        html.Div([
            html.Div([dcc.Graph(figure=fig_comp_pot)], style={**CARD_STYLE,"flex":"1"}),
            html.Div([dcc.Graph(figure=fig_scatter)],  style={**CARD_STYLE,"flex":"1"}),
        ], style={"display":"flex","gap":"16px"}),
    ], style=PAGE_STYLE)

In [ ]:
def page_frecuencia():
    fig_freq = go.Figure()
    fig_freq.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["frecuencia_media"],
        mode="lines", name="Frecuencia Media",
        line=dict(color=COLORS["accent4"], width=2), fill="tozeroy"))
    apply_template(fig_freq)
    fig_freq.update_layout(title="Evolucion de la Frecuencia Media", height=380,
        xaxis_title="Numero de registro", yaxis_title="Frecuencia (Hz)")

    fig_presion = go.Figure()
    fig_presion.add_trace(go.Scatter(x=df_line_sample["indice"], y=df_line_sample["max_presion"],
        mode="lines", name="Presion Maxima",
        line=dict(color=COLORS["accent2"], width=2), fill="tozeroy"))
    apply_template(fig_presion)
    fig_presion.update_layout(title="Evolucion de la Presion Maxima", height=380,
        xaxis_title="Numero de registro", yaxis_title="Presion (bar)")

    df_sample = df.sample(min(800, len(df)))
    fig_corr = px.scatter(df_sample, x="frecuencia_media", y="potencia_media",
        color="media_temperatura", color_continuous_scale="Reds",
        title="Frecuencia vs Potencia")
    apply_template(fig_corr)
    fig_corr.update_layout(height=320)

    return html.Div([
        html.H2("Frecuencia y Presion", style={"margin": "0 0 20px 0"}),
        html.Div([
            kpi_card("Frecuencia media", f"{df['frecuencia_media'].mean():.3f}", "Hz", COLORS["accent4"]),
            kpi_card("Frecuencia max",   f"{df['frecuencia_media'].max():.2f}",  "Hz", COLORS["accent1"]),
            kpi_card("Presion media",    f"{df['max_presion'].mean():.4f}",      "bar",COLORS["accent2"]),
            kpi_card("Presion max",      f"{df['max_presion'].max():.4f}",       "bar",COLORS["accent2"]),
        ], style={"display":"flex","gap":"12px","flexWrap":"wrap","marginBottom":"24px"}),
        html.Div([
            html.Div([dcc.Graph(figure=fig_freq)],   style={**CARD_STYLE,"flex":"1"}),
            html.Div([dcc.Graph(figure=fig_presion)],style={**CARD_STYLE,"flex":"1"}),
        ], style={"display":"flex","gap":"16px"}),
        html.Div([dcc.Graph(figure=fig_corr)], style=CARD_STYLE),
    ], style=PAGE_STYLE)

In [ ]:
def page_alertas():
    df_alertas = df[df["alertas_potencia"] > 0].copy()
    df_alert_grouped = df.groupby(df["timestamp"].dt.floor("s"))["alertas_potencia"].max().reset_index()
    df_alert_grouped["indice"] = range(len(df_alert_grouped))
    if len(df_alert_grouped) > 300:
        df_alert_sample = df_alert_grouped.iloc[::max(1, len(df_alert_grouped)//300)]
    else:
        df_alert_sample = df_alert_grouped

    fig_alertas_time = go.Figure()
    fig_alertas_time.add_trace(go.Scatter(
        x=df_alert_sample["indice"], y=df_alert_sample["alertas_potencia"],
        mode="lines+markers", line=dict(color=COLORS["accent2"], width=2),
        marker=dict(size=4), fill="tozeroy", name="Alertas"))
    apply_template(fig_alertas_time)
    fig_alertas_time.update_layout(title="Evolucion de Alertas de Potencia", height=350,
        xaxis_title="Intervalo de tiempo (agrupado)", yaxis_title="Nivel de Alerta")

    fig_compare = go.Figure()
    datos_sin = df[df["alertas_potencia"] == 0]["potencia_media"]
    if len(datos_sin) > 0:
        fig_compare.add_trace(go.Box(y=datos_sin, name="Sin Alertas",
            marker_color=COLORS["accent4"], boxmean=True))
    if len(df_alertas) > 0:
        fig_compare.add_trace(go.Box(y=df_alertas["potencia_media"], name="Con Alertas",
            marker_color=COLORS["accent2"], boxmean=True))
    apply_template(fig_compare)
    fig_compare.update_layout(title="Potencia: Con vs Sin Alertas", height=320, yaxis_title="Potencia Media (W)")

    if len(df_alertas) > 0:
        top = df.nlargest(10, "alertas_potencia")[
            ["timestamp","potencia_media","media_temperatura","alertas_potencia"]].copy()
        top["timestamp"] = top["timestamp"].astype(str)
        top = top.round(2)
        tabla_alertas = dash_table.DataTable(
            data=top.to_dict("records"),
            columns=[{"name": c, "id": c} for c in top.columns],
            style_table={"overflowX": "auto"},
            style_cell={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                         "border": f"1px solid {COLORS['border']}", "fontFamily": "Arial, sans-serif",
                         "fontSize": "11px", "padding": "6px 10px"},
            style_header={"backgroundColor": "#D40000", "color": "white", "fontWeight": "700"},
        )
    else:
        tabla_alertas = html.P("No hay alertas registradas",
            style={"color": COLORS["subtext"], "textAlign": "center", "padding": "20px"})

    pct_alertas = (len(df_alertas) / len(df) * 100) if len(df) > 0 else 0
    pot_alerta  = df_alertas["potencia_media"].mean() if len(df_alertas) > 0 else 0
    pot_str     = f"{pot_alerta:.1f}" if len(df_alertas) > 0 else "-"

    return html.Div([
        html.H2("Analisis de Alertas", style={"margin": "0 0 20px 0"}),
        html.Div([
            kpi_card("Registros con alertas",  f"{len(df_alertas):,}",              "", COLORS["accent2"]),
            kpi_card("% tiempo con alertas",   f"{pct_alertas:.2f}",                "%",COLORS["accent2"]),
            kpi_card("Alertas maximas",        f"{int(df['alertas_potencia'].max())}","",COLORS["accent1"]),
            kpi_card("Potencia media (alerta)", pot_str,                             "W",COLORS["accent4"]),
        ], style={"display":"flex","gap":"12px","flexWrap":"wrap","marginBottom":"24px"}),
        html.Div([dcc.Graph(figure=fig_alertas_time)], style=CARD_STYLE),
        html.Div([
            html.Div([dcc.Graph(figure=fig_compare)], style={**CARD_STYLE,"flex":"1"}),
            html.Div([
                html.H4("Top 10 Alertas mas altas",
                        style={"color": COLORS["text"], "margin": "0 0 12px 0", "fontSize": "14px"}),
                tabla_alertas,
            ], style={**CARD_STYLE,"flex":"1"}),
        ], style={"display":"flex","gap":"16px"}),
    ], style=PAGE_STYLE)

In [ ]:
def page_datos():
    df_show = df.copy()
    numeric_cols = df_show.select_dtypes(include=["float64","int64"]).columns
    df_show[numeric_cols] = df_show[numeric_cols].round(4)
    df_show["timestamp"] = df_show["timestamp"].astype(str)

    stats = [
        f"Total registros: {len(df):,}",
        f"Desde: {df['timestamp'].min()}",
        f"Hasta: {df['timestamp'].max()}",
        f"Columnas: {len(df.columns)}",
    ]
    return html.Div([
        html.H2("Datos Brutos", style={"margin": "0 0 20px 0"}),
        html.Div([
            html.Div([
                html.Span(s, style={"color": COLORS["subtext"], "fontSize": "13px", "marginRight": "24px"})
                for s in stats
            ], style={"marginBottom": "16px", "display": "flex", "flexWrap": "wrap", "gap": "16px"}),
            dash_table.DataTable(
                data=df_show.to_dict("records"),
                columns=[{"name": c, "id": c} for c in df_show.columns],
                page_size=25,
                filter_action="native",
                sort_action="native",
                style_table={"overflowX": "auto", "height": "60vh"},
                style_cell={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                             "border": f"1px solid {COLORS['border']}", "fontFamily": "Arial, sans-serif",
                             "fontSize": "11px", "padding": "6px 10px", "textAlign": "left"},
                style_header={"backgroundColor": "#D40000", "color": "white",
                               "fontWeight": "700", "border": f"1px solid {COLORS['border']}",
                               "textTransform": "uppercase"},
                style_data_conditional=[
                    {"if": {"filter_query": "{alertas_potencia} > 0"},
                     "backgroundColor": "rgba(164,0,0,0.15)", "color": COLORS["accent2"]},
                    {"if": {"row_index": "odd"}, "backgroundColor": "rgba(22,27,34,0.4)"},
                ],
                export_format="csv",
                export_headers="display",
            ),
        ], style=CARD_STYLE),
    ], style=PAGE_STYLE)

print("Funciones de telemetria listas")

## 8. Dashboard combinado

Ejecuta esta celda y abre **http://localhost:8058**

- Tab izquierda: EDA Compresores A-D (tema claro, filtros dinamicos)
- Tab derecha: Telemetria Industrial (tema oscuro, 6 sub-pestanas)

In [ ]:
# ================================================================
#  DASHBOARD COMBINADO  —  puerto 8058
# ================================================================

app = Dash(__name__, suppress_callback_exceptions=True)

COMPRESORES = sorted(vdf_comp["Compresor"].unique())
FREQ_MIN    = int(np.floor(vdf_comp["Frecuencia"].min()))
FREQ_MAX    = int(np.ceil(vdf_comp["Frecuencia"].max()))

# Estilos compresores (tema claro)
COMP_CARD  = {"backgroundColor": "white", "padding": "16px",
              "borderRadius": "12px", "boxShadow": "0 2px 8px rgba(0,0,0,0.08)"}
COMP_GRAPH = {**COMP_CARD, "marginBottom": "20px"}

def comp_kpi(title, vid, sub):
    return html.Div([
        html.P(title, style={"margin": 0, "color": "#555", "fontSize": "12px"}),
        html.H3(id=vid, style={"margin": "4px 0", "color": "#D40000", "fontSize": "22px"}),
        html.P(sub,   style={"margin": 0, "color": "#999", "fontSize": "11px"}),
    ], style=COMP_CARD)

# Tab 1: Compresores
tab_comp_content = html.Div([
    html.Div(
        style={"display": "grid", "gridTemplateColumns": "1fr 2fr",
               "gap": "20px", "marginBottom": "20px"},
        children=[
            html.Div(style=COMP_GRAPH, children=[
                html.Label("Compresores:",
                           style={"fontWeight": "bold", "display": "block", "marginBottom": "8px"}),
                dcc.Dropdown(
                    id="comp-filter",
                    options=[{"label": c, "value": c} for c in COMPRESORES],
                    value=COMPRESORES, multi=True, clearable=False,
                ),
            ]),
            html.Div(style=COMP_GRAPH, children=[
                html.Label("Rango de frecuencia (%):",
                           style={"fontWeight": "bold", "display": "block", "marginBottom": "8px"}),
                dcc.RangeSlider(
                    id="freq-slider",
                    min=FREQ_MIN, max=FREQ_MAX, value=[FREQ_MIN, FREQ_MAX],
                    marks={i: str(i) for i in range(0, 101, 10)},
                    tooltip={"placement": "bottom", "always_visible": True},
                ),
            ]),
        ],
    ),
    html.Div(
        style={"display": "grid", "gridTemplateColumns": "repeat(5, 1fr)",
               "gap": "14px", "marginBottom": "22px"},
        children=[
            comp_kpi("Registros filtrados", "kpi-n",    "tras filtro"),
            comp_kpi("Potencia media",       "kpi-pot",  "W medidos"),
            comp_kpi("Error abs. medio",     "kpi-err",  "W de desviacion"),
            comp_kpi("Frecuencia media",     "kpi-frec", "% operacion"),
            comp_kpi("% anomalias",          "kpi-anom", "criterio IQR"),
        ],
    ),
    html.Div(
        style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "20px"},
        children=[
            html.Div(dcc.Graph(id="g-media"),     style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-scatter"),   style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-violin"),    style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-tendencia"), style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-dist"),      style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-anom"),      style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-corr"),      style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-env"),       style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-freqhist"),  style=COMP_GRAPH),
            html.Div(dcc.Graph(id="g-freqpot"),   style=COMP_GRAPH),
        ],
    ),
    html.H4("Tabla resumen por compresor",
            style={"marginTop": "28px", "color": "#D40000"}),
    dash_table.DataTable(
        id="tabla-comp",
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "center", "padding": "8px", "fontSize": "13px"},
        style_header={"backgroundColor": "#D40000", "color": "white", "fontWeight": "bold"},
        style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f4f7fb"}],
    ),
], style={"backgroundColor": "#f0f2f5", "padding": "20px"})

# Tab 2: Telemetria
tab_telem_content = html.Div([
    html.Div(
        style={"backgroundColor": COLORS["card"],
               "borderBottom": f"1px solid {COLORS['border']}",
               "padding": "10px 24px"},
        children=[
            html.Span("TELEMETRIA INDUSTRIAL", style={
                "color": COLORS["accent1"], "fontWeight": "700",
                "fontFamily": "Arial, sans-serif", "fontSize": "15px", "letterSpacing": "2px",
            }),
        ],
    ),
    dcc.Tabs(
        id="telem-subtabs",
        value="t-resumen",
        style={"fontFamily": "Arial, sans-serif"},
        colors={"border": COLORS["border"], "primary": COLORS["accent1"], "background": COLORS["card"]},
        children=[
            dcc.Tab(label="Resumen",     value="t-resumen",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent1']}", "fontWeight": "bold"}),
            dcc.Tab(label="Temperatura", value="t-temp",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent2']}", "fontWeight": "bold"}),
            dcc.Tab(label="Potencia",    value="t-potencia",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent1']}", "fontWeight": "bold"}),
            dcc.Tab(label="Frecuencia",  value="t-frecuencia",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent4']}", "fontWeight": "bold"}),
            dcc.Tab(label="Alertas",     value="t-alertas",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent2']}", "fontWeight": "bold"}),
            dcc.Tab(label="Datos",       value="t-datos",
                    style={"backgroundColor": COLORS["card"], "color": COLORS["subtext"], "border": "none"},
                    selected_style={"backgroundColor": COLORS["bg"], "color": COLORS["text"],
                                    "borderTop": f"3px solid {COLORS['accent1']}", "fontWeight": "bold"}),
        ],
    ),
    html.Div(id="telem-content", style={"backgroundColor": COLORS["bg"]}),
], style={"backgroundColor": COLORS["bg"], "minHeight": "600px"})

# Layout principal
app.layout = html.Div(
    style={"fontFamily": "Arial"},
    children=[
        html.Div(
            style={"backgroundColor": "#D40000", "padding": "20px 28px 16px 28px"},
            children=[
                html.H2("Dashboard Reto 8: EDA Compresores + Telemetria Industrial",
                        style={"textAlign": "center", "color": "white", "margin": 0, "fontSize": "22px"}),
                html.P("Analisis exploratorio de compresores industriales y monitorizacion de trafico de red",
                       style={"textAlign": "center", "color": "#D9D9D9", "margin": "6px 0 0 0", "fontSize": "13px"}),
            ],
        ),
        dcc.Tabs(
            id="main-tabs",
            value="tab-comp",
            style={"marginBottom": "0"},
            colors={"border": "#ddd", "primary": "#D40000", "background": "#f0f2f5"},
            children=[
                dcc.Tab(label="EDA Compresores A-D", value="tab-comp",
                        children=tab_comp_content,
                        style={"fontWeight": "500"},
                        selected_style={"fontWeight": "bold", "borderTop": "3px solid #D40000"}),
                dcc.Tab(label="Telemetria Industrial", value="tab-telem",
                        children=tab_telem_content,
                        style={"fontWeight": "500"},
                        selected_style={"fontWeight": "bold", "borderTop": "3px solid #A30000"}),
            ],
        ),
    ],
)


# Callback sub-pestanas Telemetria
@app.callback(Output("telem-content", "children"), Input("telem-subtabs", "value"))
def update_telem_page(tab):
    if tab == "t-temp":         return page_temperatura()
    elif tab == "t-potencia":   return page_potencia()
    elif tab == "t-frecuencia": return page_frecuencia()
    elif tab == "t-alertas":    return page_alertas()
    elif tab == "t-datos":      return page_datos()
    else:                       return page_resumen()


# Callback Compresores
@app.callback(
    [Output("kpi-n",      "children"), Output("kpi-pot",   "children"),
     Output("kpi-err",    "children"), Output("kpi-frec",  "children"),
     Output("kpi-anom",   "children"),
     Output("g-media",    "figure"),   Output("g-scatter",   "figure"),
     Output("g-violin",   "figure"),   Output("g-tendencia", "figure"),
     Output("g-dist",     "figure"),   Output("g-anom",      "figure"),
     Output("g-corr",     "figure"),   Output("g-env",       "figure"),
     Output("g-freqhist", "figure"),   Output("g-freqpot",   "figure"),
     Output("tabla-comp", "data"),     Output("tabla-comp",  "columns")],
    [Input("comp-filter", "value"), Input("freq-slider", "value")],
)
def update_comp(comps, freq_range):
    comps = comps or COMPRESORES
    fmin, fmax = freq_range
    dff  = vdf_comp[vdf_comp["Compresor"].isin(comps) & vdf_comp["Frecuencia"].between(fmin, fmax)].copy()
    dff0 = df_comp[df_comp["Compresor"].isin(comps)  &  df_comp["Frecuencia"].between(fmin, fmax)].copy()
    kpis = [
        f"{len(dff):,}",
        f"{dff['Potencia_Medida'].mean():.2f} W",
        f"{dff['Error_Abs'].mean():.3f} W",
        f"{dff['Frecuencia'].mean():.1f} %",
        f"{dff0['Anomalia'].mean() * 100:.3f} %",
    ]
    figs = [
        fig_media_potencias(dff),
        fig_scatter_medida_estimada(dff),
        fig_error_violin(dff),
        fig_tendencia_frecuencia(dff),
        fig_distribucion_potencia(dff),
        fig_anomalias(dff0),
        fig_correlacion(dff),
        fig_presion_temp(dff),
        fig_frecuencia_hist(dff),
        fig_frecuencia_potencia(dff),
    ]
    resumen = compute_summary(dff0)
    return (
        kpis + figs +
        [resumen.to_dict("records"),
         [{"name": c, "id": c} for c in resumen.columns]]
    )


app.run(debug=True, port=8058)

## 9. Notas de uso

### Tab 1 — EDA Compresores
- Usa el desplegable para seleccionar uno o mas compresores.
- Desliza el rango de frecuencia para filtrar los datos (los 10 graficos y los KPIs se actualizan en tiempo real).
- La tabla inferior muestra el resumen estadistico del subconjunto filtrado.

### Tab 2 — Telemetria Industrial
- Navega entre las 6 sub-pestanas para explorar temperatura, potencia, frecuencia/presion, alertas y datos brutos.
- En la pestana **Datos** puedes filtrar columnas y exportar a CSV.

### Exportacion de PNGs
- Ejecuta la **celda 6** (Exportar PNGs) para guardar los 10 graficos de compresores en `plantilla_latex/eda_figures/`.
- Requiere `pip install kaleido`.